In [1]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from pprint import pprint

# ========================== CONFIG ==========================
COLLECTION_NAME = "python_coding_standards"
DB_PATH = "./chroma_python_standards_db2"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

def query_chroma(query_text: str, n_results: int = 5):
    """
    Query the Python Coding Standards knowledge base
    """
    # Load embedding function (same model used during ingestion)
    embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)

    # Connect to existing ChromaDB
    client = chromadb.PersistentClient(path=DB_PATH)
    
    collection = client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_function
    )

    print(f"🔍 Querying: '{query_text}'")
    print(f"📊 Top {n_results} results:\n")

    # Perform similarity search
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]   # You can also include embeddings if needed
    )

    # Display results nicely
    for i in range(len(results['documents'][0])):
        print(f"{'='*80}")
        print(f"Result {i+1} | Score: {1 - results['distances'][0][i]:.4f} "
              f"(Distance: {results['distances'][0][i]:.4f})")
        print(f"Section   : {results['metadatas'][0][i]['section_title']}")
        print(f"Header Level : {results['metadatas'][0][i].get('header_level', 'N/A')}")
        print(f"Parent    : {results['metadatas'][0][i].get('parent_section', 'N/A')}")
        print("-" * 80)
        
        # Print first 500 characters of content (clean preview)
        content_preview = results['documents'][0][i][:500]
        if len(results['documents'][0][i]) > 500:
            content_preview += "..."
        print(content_preview)
        print()

    return results


# # ========================== EXAMPLE QUERIES ==========================
# if __name__ == "__main__":
#     print("🤖 Python Coding Standards RAG Query Tool\n")

#     example_queries = [
#         "What are the naming conventions for classes and functions?",
#         "What is the recommended maximum line length?",
#         "How should I handle imports in Python files?",
#         "Explain the error handling best practices",
#         "What tools are recommended for code formatting and linting?",
#         "Show me the Python coding workflow diagram",
#         "What is the docstring format we should follow?",
#         "How to write unit tests according to standards?"
#     ]

#     for idx, q in enumerate(example_queries, 1):
#         print(f"\n{'#'*80}")
#         print(f"Example Query {idx}: {q}")
#         print(f"{'#'*80}")
#         query_chroma(q, n_results=3)
#         input("\nPress Enter for next query...")   # Remove this line if you don't want pause

In [5]:
# retrieve_results = query_chroma("What are the naming conventions for classes and functions?", n_results=3)

In [7]:
# print(f"Type of retrieve_results: {type(retrieve_results)}")
# print(f"Keys in retrieve_results: {retrieve_results.keys()}")

In [6]:
# print(f"Retrieve_results: {retrieve_results}")

In [ ]:
# ========================== CONFIG ==========================
GROQ_API_KEY = ""
MODEL_NAME = "llama-3.1-8b-instant"

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# ========================== LLM ==========================
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=MODEL_NAME,
    temperature=0
)

# ========================== PROMPT ==========================
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert Python coding assistant.

Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""
)

# ========================== RAG FUNCTION ==========================
def rag_query(question: str, n_results: int = 3):
    # Step 1: Retrieve from Chroma
    results = query_chroma(question, n_results=n_results)

    # Step 2: Extract documents
    docs = results["documents"][0]
    metadatas = results["metadatas"][0]

    # Step 3: Build context
    context_chunks = []
    for doc, meta in zip(docs, metadatas):
        chunk = f"""
Section: {meta.get('section_title')}
Content:
{doc}
"""
        context_chunks.append(chunk)

    context = "\n\n".join(context_chunks)

    # Step 4: Create prompt
    final_prompt = rag_prompt.format(
        context=context,
        question=question
    )
    # print(f"final prompt:\n{final_prompt}\n")

    # Step 5: Call LLM
    response = llm.invoke(final_prompt)

    print(f"LLM Response:\n{response.content}\n")

    return response.content

In [14]:
a1 = rag_query("Give me development workflow diagram", n_results=3)

🔍 Querying: 'Give me development workflow diagram'
📊 Top 3 results:

Result 1 | Score: 0.4431 (Distance: 0.5569)
Section   : 12.3 Tools Execution Flow (Simple ASCII)
Header Level : 3
Parent    : 12.2 Code Review Process (Simple ASCII Flow)
--------------------------------------------------------------------------------
Developer Saves File
```ascii
|
v
Pre-commit Hooks Trigger
|
v
+--> Black (Formatting)
|
v
+--> Ruff (Linting & Auto-fix)
|
v
+--> isort (Sort Imports)
|
v
+--> mypy (Type Checking)
|
v
All Passed? ----> No ----> Fix Issues
|
Yes
|
v
Ready for Commit / PR
```

---

**Related Pages:**

* [PEP 8 – Style Guide for Python Code](https://peps.python.org/pep-0008/)
* [Google Python Style Guide](https://google.github.io/styleguide/pyguide.html)
* [Git Workflow Standards](/display/ENG/Git+Workflow+Standa...

Result 2 | Score: 0.4118 (Distance: 0.5882)
Section   : 12.1 Development Workflow (ASCII Flow Diagram)
Header Level : 3
Parent    : Python Coding Workflow
-------------------

In [10]:
print(a1)

```ascii
 +-------------------+
| Start: New Task |
+-------------------+
|
v
+-----------------------------+
| Write Code (Follow Standards) |
+-----------------------------+
|
v
+-----------------------------+
| Run Pre-commit Hooks |
| (Black + Ruff + isort + mypy)|
+-----------------------------+
|
v
+-----------------------------+
| Write / Update Unit Tests |
+-----------------------------+
|
v
+-----------------------------+ +-----------------+
| Run pytest Locally |<----| Tests Fail |
+-----------------------------+ +-----------------+
| ^
| |
v |
+-----------------------------+ |
| All Tests Pass? |-----------+
+-----------------------------+
| |
| Yes | No
v |
+-------------------+ |
| Create Pull Request | |
+-------------------+ |
| |
v |
+-----------------------------+
| Automated CI Checks Run |
+-----------------------------+
|
v
+-----------------------------+ +-----------------+
| Code Review by Peer |<----| Review Failed |
+-----------------------------+ +------------